In [ ]:
import time
import random
import json
import numpy as np
from dotenv import load_dotenv
from google import genai
from google.genai import types
from google.genai import errors
from dataclasses import dataclass, field

In [173]:
load_dotenv()
client = genai.Client()

### Gemini api call

In [174]:
@dataclass
class API_Configs:
    model_config = types.GenerateContentConfig(
    system_instruction="Respond STRICTLY with JSON format. Do not add markdown fences.",
    max_output_tokens=8192,
)
    
    model_tiers = {
    "high": {"model_name": "gemini-3.5-flash"},
    "medium": {"model_name": "gemini-2.5-flash"},
    "low": {"model_name": "gemini-3.1-flash-lite"}
}

In [175]:
@dataclass
class Retry_Config:
    max_attempts: int = 3
    base_delay: float = 1.0
    max_delay: float = 30.0

    retry_status_codes = (429, 500, 503, 504)

In [176]:
class Call_Mode:
    stream_request = "stream_request"
    embed_request = "embed_request"

In [177]:
def _handle_retry_or_raise(e, attempt, model_name):
    # Refers to the Tuple of status codes that can be retried in Retry_Config.
            if e.code in Retry_Config.retry_status_codes:
                if attempt == Retry_Config.max_attempts:
                    print("Max attempts reached.")
                    raise e

                # Exponential backoff, i should probably add a way to check the server header for the exact time needed if it exists later (keyword: *later*)
                new_delay = random.uniform(0.5, 1.5) * Retry_Config.base_delay * (2 ** (attempt - 1))
                clamped_delay = min(new_delay, Retry_Config.max_delay)

                print(f" Code {e.code}. Backing off. Waiting {clamped_delay:.2f}s...")
                time.sleep(clamped_delay)
                return

            else:
                status_reason = getattr(e, "status", "UNKNOWN")
                
                if e.code == 404 or status_reason == "NOT_FOUND": # Where is this error? i cant find it? Muhehehehehehhe
                    print(f"Model Not Found [404]: '{model_name}' does not exist or is deprecated.")
                    
                elif e.code in (401, 403) or status_reason == "PERMISSION_DENIED":
                    print(f"Authentication Failed [{e.code}]: Check your API key, project quotas, or permissions.")
                    
                elif e.code == 400 or status_reason == "INVALID_ARGUMENT":
                    print(f"Invalid Argument [400]: Malformed structure or prompt content limits exceeded.")
                    
                else:
                    print(f"Non-retryable error [{status_reason}] ({e.code}): {e.message}")
                    
                raise e

In [178]:
def _call_stream(model_name, prompt, config):
    for attempt in range(1, Retry_Config.max_attempts + 1):
        try:
            if attempt > 1:
                print(f"Retry attempt {attempt}/{Retry_Config.max_attempts}...")
            else:
                print("Sending initial request...")

            response = client.models.generate_content_stream(
                model=model_name, contents=prompt, config=config
            )
            for chunk in response:
                yield chunk
            return
        except errors.APIError as e:
            _handle_retry_or_raise(e, attempt, model_name)
            continue

In [179]:
def _call_embedding(model_name, prompt, config):
    for attempt in range(1, Retry_Config.max_attempts + 1):
        try:
            if attempt > 1:
                print(f"Retry attempt {attempt}/{Retry_Config.max_attempts}...")
            else:
                print("Sending initial request...")

            return client.models.embed_content(
                model=model_name, contents=prompt
            )
        except errors.APIError as e:
            _handle_retry_or_raise(e, attempt, model_name)
            continue

In [180]:
def execute_call(model_priority, prompt, config, mode):

    tier = API_Configs.model_tiers.get(model_priority.lower(), API_Configs.model_tiers["medium"])
    

    mode = mode
    for attempt in range (1, Retry_Config.max_attempts + 1):

        # Api call Section
        if mode == Call_Mode.stream_request:

            model_name = tier["model_name"]
            return _call_stream(model_name , prompt, config)
                        
        elif mode == Call_Mode.embed_request:
            
            model_name = "gemini-embedding-2"
            return _call_embedding(model_name, prompt, config)
            

In [181]:
def get_structured_output(model_priority, prompt, config, mode):

    response = execute_call(model_priority = model_priority, prompt = prompt, config = config, mode = mode)

    if mode == Call_Mode.stream_request:
        full_response = "".join(getattr(chunk, "text", "") or "" for chunk in response)
        data = json.loads(full_response)
    elif mode == Call_Mode.embed_request:
        full_response = response
        data = full_response

    return data

In [182]:
#prompt = "You shall henceforth and at once provide me with TEN of the greatest most glorified spaghettie recipes that have crossed your existence."
prompt = "Say hello in 10 random languages and state which language it is"

In [183]:
result = get_structured_output(model_priority = "high", prompt = prompt, config = API_Configs.model_config, mode = Call_Mode.embed_request)

Sending initial request...


In [188]:
embeddings_list = result.embeddings[0].values
print(embeddings_list)

[-0.038926817, 0.009350159, 0.0038492964, -0.01204189, -0.010059934, 0.0073283976, 0.010920233, 0.03887873, 0.029846812, -0.048796862, -0.020647032, 0.010505356, 0.016016584, 0.012763062, -0.0053512687, -0.001811273, 0.014062244, -0.0035171974, -0.0054456857, -0.020351881, -0.004036916, -0.01878262, 0.0008650413, 0.029505022, -0.024580175, 0.0027447084, 0.005145717, -0.0038806205, 0.007102228, 0.11364401, -0.03480317, -0.022135552, 0.028202845, 0.0010981143, 0.0019960264, -0.037581716, 0.018722422, -0.01510524, -0.009664555, -0.0044340473, 0.037570093, -0.008948366, 0.004016835, 0.012274271, -0.010912961, -0.009047351, -0.004271744, 0.006910482, 0.024013711, -0.007922557, 0.0016624978, 0.0037666615, -0.013632845, -0.014871564, -0.0041538207, -0.029904319, 0.0058512096, -0.007960975, -0.009550665, 0.014071477, 0.02626866, -0.015410836, 0.019980742, 0.0005571471, -0.014325042, -0.0036303964, 0.023246897, 0.008096438, 0.006385071, -0.019306205, 0.007567568, -0.005270307, -0.021486066, -0.

In [184]:
print(json.dumps(result, indent=4))

TypeError: Object of type EmbedContentResponse is not JSON serializable